In [ ]:
%load_ext autoreload
%autoreload 2

# Aserver AGen extension

- so far, pya.Aserver was only capable of playing asigs
- to play AGens, so far they had to be rendered&played via `ag.gen_asig().play()`
- with this Aserver extension, it is possible to dispatch agens directly.
- Technically, aserver sorts dispatched items in a onset-sorted list
  - when the time has come, (in case of asigs) snippets of size block size are
    taken from the buffer and merged to the output: the summed output is played
  - for Agens, the logic is different: in the server, the next blocksize samples
    are computed (directly), allowing
    - continuously rendering synths (resp. AGens)
    - realtime modulation of synths
    - memory-saving (yet computationally probably more expensive) synthesis
- The current implementation has to be improved. Todos are:
  - processing of multi-channel AGens (>2, resp. >output bus) needs implementation and checks
  - so far the rate argument (to resample) is not implemented. It would be nice
    to automatically resample rendering within aserver
  - load-monitoring could be implemented and deactivate agens that would cause
    chopping aserver operation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

import time

# import pyamapping as pam
from pya import startup, device_info
from pya.agen.core import stereo, multi_channel
from pya.agen.lib import Line, SinOsc, WhiteNoise, expand_channels

mpl.rcParams["figure.figsize"] = (9, 3)

In [ ]:
device_info();

In [ ]:
s = startup() # specify/set device if needed

## experiment with chunkwise computation of AGens

before modifying Aserver, let's first check block-wise generation, as it should later be done on dispatched AGens.

In [ ]:
idx = 0

In [ ]:
# check howto compute consequtive sample blocks
# execute repeatedly to see phase continuation
if idx == 0:
    ag1 = SinOsc(SinOsc(100) * 1000 + 1500)
    ag1.create_graph()
bs = 256
d = ag1.generate(bs, idx, 0)
idx += bs
plt.plot(d);

Disclaimer: 
- The above will only work if the AGens sample rate matches the 
  Aserver's samplerate and if play rate=1
- I checked ResampleGen but this is quite limited as it only supports situations 
  where one by the other is an integer.
- Fine below a better solution for more flexible resampling. This is a candidate
  to be implemented deep into pya - it could also solve the upsampling problem
  mentioned in our AMICAD 2025 paper. 

In [ ]:
from pya.agen.lib import ResampleGen
ResampleGen(SinOsc(10, phase=0.0, sr=40), sr=80, downsample_children=True).gen_asig(seconds=0.25).plot(marker=".")

To create a solution that works generally let's use interp (like in Asig.resample()

- I want the rate argument r to effectively play sr/r of the input samples per second
- yet at the moment I implicitly assume that the sampling rate of the agen matches the one of the aserver.
- if this is not the case, it would require a resample anyway.
- Assume we have
  - Aserver sampling rate srs 
  - AGen sampling rate sra, 
  - .play() rate argument r
  - blocksize of server bs

In [ ]:
# test AGen to work with:
ag1 = SinOsc(90, phase=0.5, sr=1000) * Line(1, 0, 0.1, sr=1000)

# parameters
agen = ag1 
server_sr = s.sr
bs = s.bs
agen_sr = agen.sr
rate = 1.5

# initialize generation (to be done on dispatching)
ch = 0
agen_pos = 0
agen_latest_idx = 0
agen_sample_incr = agen_sr / server_sr * rate
agen_block_increment = agen_sample_incr * bs
agen_needed_samples = int(agen_block_increment + 1)

%matplotlib inline
plt.close("all")
result = []

execute the cell below a couple of times and see how rendered samples (blue dots) 
are a correct linear interpolation between the input samples (red dots)

In [ ]:
# on each generation cycle for blocksize samples do this:
agen_pos_end = agen_pos + bs * agen_sample_incr

# for all channels (add loop over ch here)
agen.generate(agen_needed_samples+1, agen_latest_idx, channel=ch)

# take values for interpolation from cache
agen_sig_val = agen.states[ch].get_from_cache(sample_count=agen_needed_samples+1, start=int(agen_pos))
npoints = agen_sig_val.shape[0]
agen_sig_pos = np.arange(int(agen_pos), int(agen_pos) + npoints)

if npoints > 0: # if there is still agen data
    dest_pos = np.arange(agen_pos, agen_pos + agen_block_increment, agen_sample_incr)
    dest_sig = np.interp(dest_pos, agen_sig_pos, agen_sig_val)
    result.append(dest_sig)    
    # now we have our result: add dest_sig to output
    plt.plot(agen_sig_pos, agen_sig_val, "r.")
    plt.plot(dest_pos, dest_sig, "b,")
    plt.ylim(-1, 1)

    agen_latest_idx += agen_needed_samples
    agen_pos += agen_block_increment
else: 
    pass # end reached

In [ ]:
# plot the full data (appended above only for inspection)
%matplotlib widget
plt.figure()
plt.plot(np.concatenate(result), ".", ms=1);

**ToDo**: 
- integrate the above-sketched interpolation strategy into Aserver
- alternatively: use this for a novel ResampleGen and wrap the agen to be played
  with it in Aserver: more flexible, less trouble.
- maybe best: make this the default procedure to deal with up-/downsampling when
  using differing sample rates in AGens.

## First test of Aserver play_agen() / agen extension

#### Test a continuously rendering Agen (i.e. without done='stop')

In [ ]:
# as the old way / reference: play ag1 via the already possible 'Asig detour'...
ag1 = SinOsc(SinOsc(6) * 30 + 300) * 0.25
ag1.gen_asig(seconds=0.5).stereo().play(onset=0.2)

In [ ]:
# now play via `s.play_agen()``  (aka dispatch via AServer)
s.play_agen(ag1, onset=0.2, out=0) # this plays forever (until stopped via the next line)

In [ ]:
s.stop() # stop (delete all dispatched items on Aserver)

AGen.play() is a more convenient interface
- Actually AGen.play() calls Aserver.play_agen()

In [ ]:
ag1.play(onset=0.2)
time.sleep(1)
s.stop()

#### Test a finite Agen (i.e. one that ends after some time)

In [ ]:
ag2 = WhiteNoise() * Line(0.5, 0, 0.5)
ag2.play(onset=0.2)

In [ ]:
s.play_agen(ag2, onset=0)

In [ ]:
ag2.play(onset=0) # now play via the AGen.play(), which calls s.play_agen() to dispatch

#### Test realtime modulation of a playing AGen 

In [ ]:
p_freq, p_vib = [400], [10]
agmod = SinOsc(SinOsc(5) * p_vib + p_freq) * 0.2
s.play_agen(agmod, 0, 0)

In [ ]:
p_freq[0] *= 1.059 # feel free to execute repeatedly

In [ ]:
p_freq[0] /= 1.059 # feel free to execute repeatedly

In [ ]:
p_vib[0] *= 2  # feel free to execute repeatedly

In [ ]:
# interactive UI-based parameter adjustments
from ipywidgets import interactive
def syngui(freq=200, vib=2):
    global p_freq, p_vib
    p_freq[0], p_vib[0] = freq, vib
interactive(syngui, freq=(100, 700, 1), vib=(0, 20, 0.2))

In [ ]:
s.stop()

#### Test multichannel (resp. stereo) AGens

In [ ]:
ag2ch = SinOsc(freq=stereo(600, 800)) * expand_channels(Line(1, 0, 4.2, curve=-4), 2, 'last')

In [ ]:

ag2ch.gen_asig().play().plot(offset=2, lw=0.2) # the old way

In [ ]:
ag2ch.play() # the new (online rendering) way: hurray that works

## MouseX, MouseY - AGen sensors for real-time rendering

Now that we can render in realtime, let's create helper sensors as AGen to
modulate AGen nodes interactively. 

- For getting the Mouse position, `pyautogui` seems applicable. 
- ToDo: check whether there is a more light-weight solution

In [ ]:
import pyautogui

# Get the current mouse position
current_mouse_position = pyautogui.position()

# Print the mouse position
print(f"Current mouse position: {current_mouse_position}")

Here are two custom AGens as Realtime Sensors. 
- maybe they can be added to agen.lib later, but first they need to mature

In [ ]:
from pya.agen.core import SingleChannelGen

class MouseX(SingleChannelGen):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        import pyautogui

    def _generate_single(
        self, 
        sample_count: int, # The amount of samples that should be generated
        start: int, # The index of the first sample
    ) -> np.ndarray:
        # block_num = self.state.data.get("block_num", 0)
        # self.state.data["block_num"] = block_num + 1
        current_mouse_position = pyautogui.position()
        return np.full(sample_count, current_mouse_position.x)
    
class MouseY(SingleChannelGen):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        import pyautogui

    def _generate_single(
        self, 
        sample_count: int, # The amount of samples that should be generated
        start: int, # The index of the first sample
    ) -> np.ndarray:
        # block_num = self.state.data.get("block_num", 0)
        # self.state.data["block_num"] = block_num + 1
        current_mouse_position = pyautogui.position()
        return np.full(sample_count, current_mouse_position.y)

In [ ]:
# MouseX test: execute, then move mMuse pointer horizontally
for i in range(30):
    print("move Mouse pointer! ==> Mouse.x = ", MouseX(sr=10).gen_asig(1).sig, end='\r')
    time.sleep(0.1)

Now a first online synthesis using MouseX and MouseY

In [ ]:
agfreq = MouseX().linlin(0, 2560, 200, 400) # x for frequency
agvib = MouseY().linlin(0, 2000, 0, 20) # y for vibrato speed
(SinOsc(agfreq) * SinOsc(agvib).linlin(-1,1,0,0.5)).play();

In [ ]:
s.stop()

**Benchmarking:** 
- MouseX fills all values with current pointer.x coordinate. 
- this is executed only once per block: i.e. 
- with current Aserver defaults (s=44100, bs=512) at ~86 Hz
- Here the timing of a single call (i.e. position queried once per generate)
  - its in the order of 0.5 ms on my M2 MBP

In [ ]:
t = time.time(); MouseX().generate(512, 0, 0); print(f"used time = {(time.time() -t)* 1000:4.2f} milliseconds")